In [0]:
%pip install loguru
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


# 03 — Gold Feature Engineering: ML-Ready Feature Table

Reads validated Silver data and engineers features proven to predict
Remaining Useful Life (RUL) of rotating equipment.

| Property | Value |
|----------|-------|
| **Medallion Layer** | Gold (ML-ready, business-level) |
| **Source** | `silver_sensor_readings_validated` Delta table |
| **Sink** | `gold_features_rul` Delta table |
| **Features** | Rolling stats, lag features, rate-of-change, RUL labels |
| **Optimization** | Z-ORDER on `unit_id` for ML training query performance |
| **Unity Catalog** | `mining_ops.predictive_maintenance.gold_features_rul` |

**Mining context:** Feature windows map to operational shifts — window=5 ≈ one
shift, window=20 ≈ one maintenance cycle. RUL cap at 125 aligns with typical
haul truck engine overhaul intervals.

## 1. Dependencies

In [0]:
import sys

repo_root = "/Workspace/Users/catherine.varas.padilla@gmail.com/predictive-maintenance-lakehouse"
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
    print(f"Added to sys.path: {repo_root}")

## 2. Configuration

In [0]:
dbutils.widgets.text("silver_path", "/Volumes/workspace/default/silver/sensor_readings_validated", "Silver Delta path")
dbutils.widgets.text("gold_path", "/Volumes/workspace/default/gold/features_rul", "Gold Delta path")
dbutils.widgets.text("max_rul_cap", "125", "RUL cap value")
dbutils.widgets.text("window_sizes", "5,10,20", "Rolling window sizes (comma-separated)")
dbutils.widgets.text("lags", "1,5,10", "Lag periods (comma-separated)")

SILVER_PATH = dbutils.widgets.get("silver_path")
GOLD_PATH = dbutils.widgets.get("gold_path")
MAX_RUL_CAP = int(dbutils.widgets.get("max_rul_cap"))
WINDOW_SIZES = [int(x.strip()) for x in dbutils.widgets.get("window_sizes").split(",")]
LAGS = [int(x.strip()) for x in dbutils.widgets.get("lags").split(",")]

print(f"Silver source:  {SILVER_PATH}")
print(f"Gold sink:      {GOLD_PATH}")
print(f"RUL cap:        {MAX_RUL_CAP}")
print(f"Window sizes:   {WINDOW_SIZES}")
print(f"Lag periods:    {LAGS}")

Silver source:  /Volumes/workspace/default/silver/sensor_readings_validated
Gold sink:      /Volumes/workspace/default/gold/features_rul
RUL cap:        125
Window sizes:   [5, 10, 20]
Lag periods:    [1, 5, 10]


## 3. Imports

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp
from delta.tables import DeltaTable

from src.feature_engineering import (
    rolling_sensor_stats,
    create_lag_features,
    create_rate_of_change_features,
    compute_rul_labels,
)

spark = SparkSession.builder.getOrCreate()

## 4. Read Silver

In [0]:
silver_df = spark.read.format("delta").load(SILVER_PATH)
silver_count = silver_df.count()
print(f"Silver records loaded: {silver_count:,}")
print(f"Unique units: {silver_df.select('unit_id').distinct().count()}")
print(f"Columns: {len(silver_df.columns)}")

if silver_count == 0:
    dbutils.notebook.exit("ERROR: Silver table is empty — run notebook 02 first.")

Silver records loaded: 867
Unique units: 5
Columns: 32


## 5. Convert to Pandas for Feature Engineering

The `src.feature_engineering` module operates on pandas DataFrames.
For production scale (>10M rows), replace with PySpark window functions
(see commented examples below).

In [0]:
silver_pdf = silver_df.toPandas()
silver_pdf = silver_pdf.sort_values(["unit_id", "cycle"]).reset_index(drop=True)

sensor_cols = sorted([c for c in silver_pdf.columns if c.startswith("s") and c[1:].isdigit()])
print(f"Sensor columns ({len(sensor_cols)}): {sensor_cols}")

Sensor columns (21): ['s1', 's10', 's11', 's12', 's13', 's14', 's15', 's16', 's17', 's18', 's19', 's2', 's20', 's21', 's3', 's4', 's5', 's6', 's7', 's8', 's9']


## 6. Rolling Sensor Statistics

Computes per-unit rolling mean, std, min, max, and range for each sensor
across multiple window sizes. These capture both short-term transients
and medium-term degradation trends.

In [0]:
features_pdf = rolling_sensor_stats(
    silver_pdf,
    sensor_cols=sensor_cols,
    window_sizes=WINDOW_SIZES,
)
print(f"After rolling stats: {features_pdf.shape[1]} columns (+{features_pdf.shape[1] - silver_pdf.shape[1]} new)")

/Workspace/Users/catherine.varas.padilla@gmail.com/predictive-maintenance-lakehouse/src/feature_engineering.py:63: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[f"{col}_rolling_std_{window}"] = grouped.transform(
/Workspace/Users/catherine.varas.padilla@gmail.com/predictive-maintenance-lakehouse/src/feature_engineering.py:66: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[f"{col}_rolling_min_{window}"] = grouped.transform(
/Workspace/Users/catherine.varas.padilla@gmail.com/predictive-maintenance-lakehouse/src/featur

After rolling stats: 347 columns (+315 new)


## 7. Lag Features

Lag features capture temporal dynamics — how sensor values compare
to their recent history within each unit.

In [0]:
features_pdf = create_lag_features(
    features_pdf,
    sensor_cols=sensor_cols,
    lags=LAGS,
)
print(f"After lag features: {features_pdf.shape[1]} columns")

2026-04-19 08:29:42.372 | INFO     | src.feature_engineering:create_lag_features:181 - Lag features: 63 new columns (21 sensors × 3 lags)


After lag features: 410 columns


## 8. Rate of Change

In [0]:
features_pdf = create_rate_of_change_features(
    features_pdf,
    sensor_cols=sensor_cols,
)
print(f"After rate-of-change: {features_pdf.shape[1]} columns")

2026-04-19 08:29:42.645 | INFO     | src.feature_engineering:create_rate_of_change_features:223 - Rate-of-change: 21 new columns


After rate-of-change: 431 columns


## 9. RUL Labels

Piece-wise linear RUL: flat at `max_rul_cap` during early life, then
linearly decreasing toward failure. This focuses the model on the
degradation phase that matters for maintenance scheduling.

In [0]:
features_pdf = compute_rul_labels(
    features_pdf,
    max_rul_cap=MAX_RUL_CAP,
)
print(f"RUL range: [{features_pdf['rul'].min()}, {features_pdf['rul'].max()}]")

2026-04-19 08:29:42.886 | INFO     | src.feature_engineering:compute_rul_labels:125 - RUL labels: range [0, 125], cap=125, 5 units


RUL range: [0, 125]


## 10. Drop NaN Rows & Add Metadata

In [0]:
rows_before = len(features_pdf)
features_pdf = features_pdf.dropna().reset_index(drop=True)
rows_dropped = rows_before - len(features_pdf)

print(f"Rows before dropna: {rows_before:,}")
print(f"Rows after dropna:  {len(features_pdf):,}")
print(f"Rows dropped (NaN): {rows_dropped:,} ({rows_dropped / max(rows_before, 1) * 100:.1f}%)")

Rows before dropna: 867
Rows after dropna:  817
Rows dropped (NaN): 50 (5.8%)


## 11. Write Gold Delta

In [0]:
gold_sdf = spark.createDataFrame(features_pdf)
gold_sdf = gold_sdf.withColumn("_gold_timestamp", current_timestamp())

(
    gold_sdf.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_PATH)
)
print(f"Gold table written to {GOLD_PATH}")

Gold table written to /Volumes/workspace/default/gold/features_rul


## 12. Z-ORDER Optimization

Z-ORDER on `unit_id` co-locates all readings for a single unit,
dramatically improving ML training queries that filter by unit.

In [0]:
spark.sql(f"OPTIMIZE delta.`{GOLD_PATH}` ZORDER BY (unit_id)")
print("Z-ORDER optimization complete on unit_id.")

Z-ORDER optimization complete on unit_id.


## 13. Validate & Column Lineage

In [0]:
gold_result = spark.read.format("delta").load(GOLD_PATH)
gold_count = gold_result.count()

original_cols = [c for c in gold_result.columns if not c.startswith("_") and "_rolling_" not in c and "_lag_" not in c and "_roc" not in c and c != "rul"]
rolling_cols = [c for c in gold_result.columns if "_rolling_" in c]
lag_cols = [c for c in gold_result.columns if "_lag_" in c]
roc_cols = [c for c in gold_result.columns if c.endswith("_roc")]
meta_cols = [c for c in gold_result.columns if c.startswith("_")]

print("=" * 60)
print("GOLD FEATURE ENGINEERING SUMMARY")
print("=" * 60)
print(f"  Silver input:         {silver_count:,} records")
print(f"  Gold output:          {gold_count:,} records")
print(f"  Total columns:        {len(gold_result.columns)}")
print(f"  Column lineage:")
print(f"    Original sensors:   {len(original_cols)}")
print(f"    Rolling stats:      {len(rolling_cols)}")
print(f"    Lag features:       {len(lag_cols)}")
print(f"    Rate-of-change:     {len(roc_cols)}")
print(f"    Metadata:           {len(meta_cols)}")
print(f"    RUL label:          1")
print(f"  RUL distribution:")
print(f"    Min: {features_pdf['rul'].min()}, Max: {features_pdf['rul'].max()}, Mean: {features_pdf['rul'].mean():.1f}")
print("=" * 60)

GOLD FEATURE ENGINEERING SUMMARY
  Silver input:         867 records
  Gold output:          817 records
  Total columns:        433
  Column lineage:
    Original sensors:   27
    Rolling stats:      315
    Lag features:       63
    Rate-of-change:     21
    Metadata:           6
    RUL label:          1
  RUL distribution:
    Min: 0, Max: 125, Mean: 82.9


## 14. Sample Gold Data

In [0]:
display(gold_result.select("unit_id", "cycle", "rul", "s2", "s3", "s2_rolling_mean_5", "s2_lag_1", "s2_roc").orderBy("unit_id", "cycle").limit(20))

unit_id,cycle,rul,s2,s3,s2_rolling_mean_5,s2_lag_1,s2_roc
1,11,125,642.28,1581.75,642.2299999999999,641.71,0.5699999999999363
1,12,125,642.06,1583.41,642.146,642.28,-0.22000000000002728
1,13,125,643.07,1582.19,642.248,642.06,1.0100000000001046
1,14,125,642.35,1592.95,642.294,643.07,-0.7200000000000273
1,15,125,642.43,1583.82,642.4379999999999,642.35,0.07999999999992724
1,16,125,642.13,1587.98,642.408,642.43,-0.2999999999999545
1,17,125,642.58,1584.96,642.5120000000001,642.13,0.4500000000000455
1,18,125,642.62,1591.04,642.422,642.58,0.03999999999996362
1,19,125,641.79,1587.56,642.3100000000001,642.62,-0.8300000000000409
1,20,125,643.04,1581.11,642.432,641.79,1.25


## 15. Per-Unit Summary

In [0]:
from pyspark.sql.functions import min as spark_min, max as spark_max, avg as spark_avg, count as spark_count

unit_summary = (
    gold_result.groupBy("unit_id")
    .agg(
        spark_count("cycle").alias("total_cycles"),
        spark_min("rul").alias("min_rul"),
        spark_max("rul").alias("max_rul"),
        spark_avg("rul").alias("avg_rul"),
    )
    .orderBy("unit_id")
)
display(unit_summary)

unit_id,total_cycles,min_rul,max_rul,avg_rul
1,178,0,125,81.67977528089888
2,265,0,125,97.05283018867925
3,163,0,125,78.50306748466258
4,169,0,125,81.89349112426035
5,42,0,42,20.738095238095237
